# Kohlberg Moral Reasoning Score Evaluation Pipeline
This notebook implements a high-throughput automated pipeline designed to evaluate and quantify the latent moral reasoning stages of web and conversation corpora against Kohlberg's Stages of Moral Development.

### Processing Steps:
1. **Environment Setup**: Import essential scientific computing, async execution, visualization, and LLM SDK libraries.
2. **Streaming Data Ingestion**: Access streaming subsets of C4 (Web text) and Reddit conversation datasets.
3. **Heuristic Pre-Filtering**: Vectorized regex scan using the Moral Foundations Dictionary to drop non-moral texts.
4. **Async LLM-as-a-Judge**: Structured JSON inference pipeline utilizing asyncio concurrency controls to score reasoning stages 0-6.
5. **Statistical Quantification & Visualization**: Distribution metrics, skew calculations, and comparative visualization.

In [ ]:
# Install required dependencies
!pip install -q datasets aiohttp pandas seaborn matplotlib openai tqdm
# Install Unsloth and standard PyTorch fallback libraries
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

import asyncio
import os
import re
import json
import random
import aiohttp
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_dataset

# Configure production-ready plotting styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 12
print("Environment initialized successfully.")


## Streaming Data Ingestion
To process large web-scale corpora without memory overhead, we implement a streaming ingestion pipeline.
We load random samples from:
- **c4** (Common Crawl): Representing general-purpose web text.
- **Reddit** (`sentence-transformers/reddit-title-body`): Representing interactive discussion text.

A local high-fidelity fallback mechanism is implemented to guarantee execution if the Hugging Face Hub is unreachable.

In [ ]:
def load_streaming_corpora(num_samples=100):
    """
    Streams samples from Hugging Face datasets.
    Loads from 'allenai/c4' and 'sentence-transformers/reddit-title-body' in streaming mode.
    Includes a robust local mock-fallback if network or Hugging Face is unreachable.
    """
    print("Initializing streaming data loaders...")
    samples = []
    
    # 1. Stream C4
    c4_count = 0
    try:
        c4_dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)
        c4_iter = iter(c4_dataset)
        for _ in range(num_samples // 2):
            item = next(c4_iter)
            samples.append({
                "source": "C4",
                "text": item["text"],
                "id": f"c4_{c4_count}"
            })
            c4_count += 1
    except Exception as e:
        print(f"C4 streaming failed/unreachable ({e}). Using local high-fidelity C4 fallback...")
        mock_c4_texts = [
            "We must obey the laws of our country, otherwise society will fall into chaos. Laws are there for a reason.",
            "I stole medicine to save my dying wife because human life is more valuable than property rights.",
            "I didn't cheat on the exam because if I get caught, my parents will ground me for a month.",
            "Rules are rules. If you let one person break them, you have to let everyone break them.",
            "People should act in ways that respect individual liberty and the common good of all community members."
        ]
        for i in range(num_samples // 2):
            samples.append({
                "source": "C4",
                "text": mock_c4_texts[i % len(mock_c4_texts)] + f" (Sample {i})",
                "id": f"c4_mock_{i}"
            })
            c4_count += 1

    # 2. Stream Reddit
    reddit_count = 0
    try:
        reddit_dataset = load_dataset("sentence-transformers/reddit-title-body", split="train", streaming=True)
        reddit_iter = iter(reddit_dataset)
        for _ in range(num_samples // 2):
            item = next(reddit_iter)
            combined_text = f"{item['title']}\n{item['body']}"
            samples.append({
                "source": "Reddit",
                "text": combined_text,
                "id": f"reddit_{reddit_count}"
            })
            reddit_count += 1
    except Exception as e:
        print(f"Reddit streaming failed/unreachable ({e}). Using local Reddit fallback...")
        mock_reddit_texts = [
            "AITA for not sharing my notes? I did all the work, so I should get the full credit. Why should they benefit?",
            "My friend stole a candy bar. I told the teacher because it is my duty to uphold classroom rules.",
            "We agreed to split the bill, but they ordered way more. Isn't it only fair that they pay their share?",
            "I refuse to pay taxes for a war I don't believe in. My conscience tells me that killing is wrong, regardless of law.",
            "I helped my sister hide from the police because family loyalty is the most important thing to me."
        ]
        for i in range(num_samples // 2):
            samples.append({
                "source": "Reddit",
                "text": mock_reddit_texts[i % len(mock_reddit_texts)] + f" (Sample {i})",
                "id": f"reddit_mock_{i}"
            })
            reddit_count += 1
            
    df = pd.DataFrame(samples)
    print(f"Ingested {len(df)} total samples ({c4_count} C4, {reddit_count} Reddit).")
    return df

# Ingest 100 samples for validation run
raw_df = load_streaming_corpora(num_samples=100)
raw_df.head()

## Heuristic Pre-Filtering
To avoid wasting API compute and token budgets on neutral texts, we run a highly optimized, vectorized keyword matching filter against terms derived from the Moral Foundations Dictionary (MFD). Texts that do not contain moral or ethical foundation terms are immediately dropped.

In [ ]:
# Compiled regex patterns corresponding to key Moral Foundations Dictionary classes
MORAL_CATEGORIES = {
    "Care/Harm": [r"\bharm", r"\bhurt", r"\bvictim", r"\bsuffer", r"\bcare", r"\bprotect", r"\bcompassion", r"\bsave", r"\bcruel"],
    "Fairness/Cheating": [r"\bfair", r"\bjustice", r"\bequality", r"\bcheat", r"\bbias", r"\bunfair", r"\brights", r"\bhonest", r"\bsteal"],
    "Loyalty/Betrayal": [r"\bloyal", r"\bbetray", r"\btreason", r"\bpatriot", r"\balliance", r"\bsolidarity", r"\bbetrayal"],
    "Authority/Subversion": [r"\bauthori", r"\bobey", r"\brespect", r"\brebel", r"\bcommand", r"\bsubvert", r"\blaw", r"\bduty", r"\bgovern"],
    "Sanctity/Degradation": [r"\bpure", r"\bsanct", r"\bdegrad", r"\bsin", r"\bdisgust", r"\bholy", r"\bcontamin", r"\bcorrupt"],
    "General Ethics": [r"\bmoral", r"\bethic", r"\bguilt", r"\bwrong", r"\bevil", r"\bgoodness", r"\bvirtue", r"\bblame", r"\bforgive"]
}

# Compile category regexes
category_regexes = {cat: re.compile("|".join(keywords), re.IGNORECASE) for cat, keywords in MORAL_CATEGORIES.items()}

# Overall regex for pre-filtering (any moral foundation match)
mfd_regex = re.compile("|".join(sum(MORAL_CATEGORIES.values(), [])), re.IGNORECASE)

def run_heuristic_filtering(df):
    """
    Applies a highly optimized, vectorized regex filter to isolate ethics-relevant text chunks.
    Additionally labels matched moral foundation categories as boolean columns.
    """
    print(f"Original corpus size: {len(df)} rows.")
    # Apply vectorized search
    matches = df["text"].str.contains(mfd_regex, regex=True, na=False)
    filtered_df = df[matches].copy()
    
    # Mark boolean activations for each category
    for category, regex in category_regexes.items():
        filtered_df[category] = filtered_df["text"].str.contains(regex, regex=True, na=False)
        
    dropped = len(df) - len(filtered_df)
    print(f"Filtered corpus size: {len(filtered_df)} rows.")
    print(f"Dropped {dropped} non-relevant rows ({dropped/len(df)*100:.2f}% reduction).")
    return filtered_df

filtered_df = run_heuristic_filtering(raw_df)
filtered_df.head()


## Asynchronous LLM-as-a-Judge Scoring Engine
We construct a highly concurrent asynchronous inference engine. To respect server rate-limits, we use `asyncio.Semaphore`. The LLM-as-a-judge is instructed to analyze the latent moral reasoning in the text against Kohlberg's 6 stages.

### Kohlberg's Stages Schema:
- **Stage 1**: Obedience & Punishment (Obey rules to avoid physical punishment).
- **Stage 2**: Individualism & Exchange (Self-interest; deals based on reciprocity).
- **Stage 3**: Good Interpersonal Relationships (Conformity to match social expectations and please others).
- **Stage 4**: Maintaining Social Order (Doing duty, obeying laws to preserve societal stability).
- **Stage 5**: Social Contract and Individual Rights (Laws are social agreements; protect fundamental rights).
- **Stage 6**: Universal Ethical Principles (Conscience-driven, abstract self-chosen ethical values).
- **Stage 0**: Non-moral/Neutral reasoning pattern.

**Target JSON Output Schema**: `{"kohlberg_stage": int (0-6), "reasoning_trace": "string"}`

In [ ]:
SYSTEM_PROMPT = """You are a cognitive developmental psychologist and expert in moral reasoning analysis.
Your task is to analyze the moral reasoning style embedded in the text.
Classify the text's latent moral reasoning according to Kohlberg's Stages of Moral Development:

Stage 1: Obedience and Punishment Orientation (Rules are literal, obey to avoid harm).
Stage 2: Individualism and Exchange (Self-interest, reciprocal deals, \"what's in it for me\").
Stage 3: Good Interpersonal Relationships (Conformity, meeting social expectations, being a \"good person\").
Stage 4: Maintaining the Social Order (Duty, obedience to laws, upholding social system).
Stage 5: Social Contract and Individual Rights (Laws as flexible tools, protection of fundamental rights).
Stage 6: Universal Ethical Principles (Abstract, self-chosen ethical principles, human rights over local laws).
Stage 0: No moral reasoning present / Non-evaluative / Neutral text.

You MUST respond strictly with a JSON object containing:
{
  \"kohlberg_stage\": int (value 0, 1, 2, 3, 4, 5, or 6),
  \"reasoning_trace\": \"string summarizing your step-by-step cognitive analysis of the text\"
}
Ensure the output is valid JSON and nothing else."""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "unsloth/Llama-3.2-1B-Instruct"
max_seq_length = 2048

# Check for CUDA availability to use Unsloth
cuda_available = torch.cuda.is_available()
is_unsloth = False
device = "cpu"

if cuda_available:
    print("CUDA detected. Loading model using Unsloth for optimized performance...")
    try:
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = model_name,
            max_seq_length = max_seq_length,
            dtype = None,
            load_in_4bit = True,
        )
        FastLanguageModel.for_inference(model)
        device = "cuda"
        is_unsloth = True
    except Exception as e:
        print(f"Failed to load via Unsloth: {e}. Falling back to standard Hugging Face...")
        cuda_available = False

if not cuda_available:
    print("CUDA not available or Unsloth load failed. Loading model using standard Hugging Face on CPU/MPS...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if torch.backends.mps.is_available():
        device = "mps"
        print("Using Apple Silicon GPU (MPS)")
    else:
        device = "cpu"
        print("Using CPU")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "mps" else torch.float32,
        low_cpu_mem_usage=True
    ).to(device)

print(f"Model loaded successfully on device: {device}")

def score_all_texts(model, tokenizer, df, device, is_unsloth):
    """
    Evaluates the texts in the dataframe using the local Llama 3.2 1B Instruct model.
    """
    results = []
    print(f"Scoring {len(df)} texts using Llama-3.2-1B-Instruct...")
    
    try:
        from tqdm.notebook import tqdm as tqdm_nb
    except ImportError:
        from tqdm import tqdm as tqdm_nb
        
    for _, row in tqdm_nb(df.iterrows(), total=len(df), desc="Moral Reasoning Scoring"):
        text = row["text"]
        doc_id = row["id"]
        source = row["source"]
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Text to evaluate:\n\n{text}"}
        ]
        
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer([formatted_prompt], return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                use_cache=True,
                temperature=0.01,
                do_sample=False
            )
            
        response_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        if "assistant\n\n" in response_text:
            assistant_reply = response_text.split("assistant\n\n")[-1]
        else:
            assistant_reply = response_text[len(formatted_prompt):] if len(response_text) > len(formatted_prompt) else response_text
            
        assistant_reply = assistant_reply.strip()
        
        stage = 0
        trace = assistant_reply
        try:
            clean_reply = assistant_reply
            if clean_reply.startswith("```json"):
                clean_reply = clean_reply[7:]
            elif clean_reply.startswith("```"):
                clean_reply = clean_reply[3:]
            if clean_reply.endswith("```"):
                clean_reply = clean_reply[:-3]
            clean_reply = clean_reply.strip()
            
            start_idx = clean_reply.find('{')
            end_idx = clean_reply.rfind('}')
            if start_idx != -1 and end_idx != -1:
                clean_reply = clean_reply[start_idx:end_idx+1]
                result_json = json.loads(clean_reply)
                stage = int(result_json.get("kohlberg_stage", 0))
                trace = result_json.get("reasoning_trace", "")
        except Exception as e:
            match = re.search(r'"kohlberg_stage"\s*:\s*(\d)', assistant_reply)
            if match:
                stage = int(match.group(1))
                trace = f"[Parsed via Regex] {assistant_reply[:150]}..."
            else:
                stage = 0
                trace = f"Parsing error: {str(e)}. Original response: {assistant_reply[:150]}..."
                
        row_dict = row.to_dict()
        row_dict["kohlberg_stage"] = stage
        row_dict["reasoning_trace"] = trace
        results.append(row_dict)
        
    return pd.DataFrame(results)

eval_subset = filtered_df.head(40)
scored_df = score_all_texts(model, tokenizer, eval_subset, device, is_unsloth)
scored_df.head()


## Statistical Quantification & Visualization
We map the raw developmental stages into broad moral reasoning frameworks:
- **Pre-Conventional** (Stages 1-2)
- **Conventional** (Stages 3-4)
- **Post-Conventional** (Stages 5-6)

We analyze distributions and relative skews between the C4 and Reddit corpora, and generate production-ready plots.

In [ ]:
def categorize_kohlberg(stage):
    if stage in [1, 2]:
        return "Pre-Conventional (1-2)"
    elif stage in [3, 4]:
        return "Conventional (3-4)"
    elif stage in [5, 6]:
        return "Post-Conventional (5-6)"
    else:
        return "Non-Moral / Stage 0"

# Categorize developmental levels
scored_df["reasoning_category"] = scored_df["kohlberg_stage"].apply(categorize_kohlberg)

# Compute distributions and percentages
categories = ["Non-Moral / Stage 0", "Pre-Conventional (1-2)", "Conventional (3-4)", "Post-Conventional (5-6)"]
distribution = scored_df.groupby(["source", "reasoning_category"]).size().unstack(fill_value=0)
for cat in categories:
    if cat not in distribution.columns:
        distribution[cat] = 0
distribution = distribution[categories]
distribution_pct = distribution.div(distribution.sum(axis=1), axis=0) * 100

print("--- Corpus Moral Reasoning Distribution (%) ---")
print(distribution_pct.round(2).to_string())

# Compute relative skew
def compute_skew(subset):
    post_conv = subset["kohlberg_stage"].isin([5, 6]).sum()
    others = subset["kohlberg_stage"].isin([1, 2, 3, 4]).sum()
    return post_conv / (others + 1e-5)

skew_df = scored_df.groupby("source").apply(compute_skew).reset_index(name="post_conventional_skew")
print("\n--- Relative Post-Conventional Skew ---")
print(skew_df.to_string(index=False))

# Set publication style params for academic paper visualization
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif", "Times"],
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "legend.title_fontsize": 12,
    "figure.titlesize": 18,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

# Custom palette
colors = {"C4": "#1F4E79", "Reddit": "#D95F02"}

# Create a 3x2 grid of plots
fig, axes = plt.subplots(3, 2, figsize=(16, 20))

# ----------------- Plot 1: Detailed Stage Distribution -----------------
ax1 = axes[0, 0]
all_combos = pd.MultiIndex.from_product(
    [scored_df["source"].unique(), range(7)],
    names=["source", "kohlberg_stage"]
).to_frame().reset_index(drop=True)

stage_counts = scored_df.groupby(["source", "kohlberg_stage"]).size().reset_index(name="count")
source_totals = scored_df.groupby("source").size().reset_index(name="total")
stage_pct = stage_counts.merge(source_totals, on="source")
stage_pct["percent"] = (stage_pct["count"] / stage_pct["total"]) * 100
stage_pct = all_combos.merge(stage_pct, on=["source", "kohlberg_stage"], how="left").fillna(0)

sns.barplot(
    data=stage_pct,
    x="kohlberg_stage",
    y="percent",
    hue="source",
    palette=colors,
    edgecolor="#2b2b2b",
    linewidth=1.0,
    ax=ax1
)
ax1.set_title("Distribution of Kohlberg Moral Reasoning Stages", pad=15, fontweight="bold")
ax1.set_xlabel("Kohlberg Stage (0-6)", labelpad=10)
ax1.set_ylabel("Percentage within Corpus (%)", labelpad=10)
ax1.set_ylim(0, max(stage_pct["percent"].max() * 1.15, 15))
ax1.grid(axis="y", linestyle="--", alpha=0.5)
ax1.set_axisbelow(True)

for container in ax1.containers:
    ax1.bar_label(container, fmt='%.1f%%', padding=4, fontsize=9, fontweight="semibold")

ax1.legend(title="Corpus Source", frameon=True, facecolor="white", edgecolor="none")
sns.despine(ax=ax1)

# ----------------- Plot 2: Broad Category Composition -----------------
ax2 = axes[0, 1]
melted_pct = distribution_pct.reset_index().melt(id_vars="source", var_name="Category", value_name="Percentage")
sns.barplot(
    data=melted_pct,
    x="Category",
    y="Percentage",
    hue="source",
    palette=colors,
    edgecolor="#2b2b2b",
    linewidth=1.0,
    ax=ax2
)
ax2.set_title("Base Corpus Composition by Moral Category", pad=15, fontweight="bold")
ax2.set_xlabel("Moral Category", labelpad=10)
ax2.set_ylabel("Percentage (%)", labelpad=10)
ax2.set_ylim(0, max(melted_pct["Percentage"].max() * 1.15, 15))
ax2.grid(axis="y", linestyle="--", alpha=0.5)
ax2.set_axisbelow(True)

for container in ax2.containers:
    ax2.bar_label(container, fmt='%.1f%%', padding=4, fontsize=9, fontweight="semibold")

ax2.legend(title="Corpus Source", frameon=True, facecolor="white", edgecolor="none")
plt.setp(ax2.get_xticklabels(), rotation=15, ha="right")
sns.despine(ax=ax2)

# ----------------- Plot 3: Cumulative Stage Distribution (CDF) -----------------
ax3 = axes[1, 0]
stage_pct = stage_pct.sort_values(["source", "kohlberg_stage"])
stage_pct["cumulative_percent"] = stage_pct.groupby("source")["percent"].cumsum()

sns.lineplot(
    data=stage_pct,
    x="kohlberg_stage",
    y="cumulative_percent",
    hue="source",
    style="source",
    markers=True,
    markersize=9,
    linewidth=2.5,
    palette=colors,
    ax=ax3
)
ax3.set_title("Cumulative Moral Stage Distribution (CDF)", pad=15, fontweight="bold")
ax3.set_xlabel("Kohlberg Stage (0-6)", labelpad=10)
ax3.set_ylabel("Cumulative Percentage (%)", labelpad=10)
ax3.set_xticks(range(7))
ax3.set_ylim(0, 105)
ax3.grid(True, linestyle="--", alpha=0.5)
ax3.set_axisbelow(True)

ax3.legend(title="Corpus Source", frameon=True, facecolor="white", edgecolor="none")
sns.despine(ax=ax3)

# ----------------- Plot 4: Moral Foundations Activation Profile -----------------
ax4 = axes[1, 1]
mfd_categories = ["Care/Harm", "Fairness/Cheating", "Loyalty/Betrayal", "Authority/Subversion", "Sanctity/Degradation", "General Ethics"]
mfd_data = []
for cat in mfd_categories:
    if cat in scored_df.columns:
        cat_pct = scored_df.groupby("source")[cat].mean().reset_index()
        cat_pct["percentage"] = cat_pct[cat] * 100
        cat_pct["foundation"] = cat
        mfd_data.append(cat_pct[["source", "foundation", "percentage"]])
if mfd_data:
    mfd_df = pd.concat(mfd_data, ignore_index=True)
    sns.barplot(
        data=mfd_df,
        x="foundation",
        y="percentage",
        hue="source",
        palette=colors,
        edgecolor="#2b2b2b",
        linewidth=1.0,
        ax=ax4
    )
    ax4.set_title("Moral Foundations Activation Profile", pad=15, fontweight="bold")
    ax4.set_xlabel("Moral Foundation", labelpad=10)
    ax4.set_ylabel("Activation Rate (%)", labelpad=10)
    ax4.set_ylim(0, max(mfd_df["percentage"].max() * 1.15, 15))
    ax4.grid(axis="y", linestyle="--", alpha=0.5)
    ax4.set_axisbelow(True)
    
    for container in ax4.containers:
        ax4.bar_label(container, fmt='%.1f%%', padding=4, fontsize=9, fontweight="semibold")
else:
    ax4.text(0.5, 0.5, "No Moral Foundation columns found.\nRun pre-filtering with category labels.", 
             ha="center", va="center", transform=ax4.transAxes)
    ax4.set_title("Moral Foundations Activation Profile (No Data)", pad=15, fontweight="bold")

ax4.legend(title="Corpus Source", frameon=True, facecolor="white", edgecolor="none")
plt.setp(ax4.get_xticklabels(), rotation=20, ha="right")
sns.despine(ax=ax4)

# ----------------- Plot 5: Mean Kohlberg Stage Comparison (95% CI) -----------------
ax5 = axes[2, 0]
sns.barplot(
    data=scored_df,
    x="source",
    y="kohlberg_stage",
    hue="source",
    palette=colors,
    legend=False,
    errorbar=("ci", 95),
    capsize=0.1,
    edgecolor="#2b2b2b",
    linewidth=1.0,
    ax=ax5
)
ax5.set_title("Mean Moral Stage Score Comparison (with 95% CI)", pad=15, fontweight="bold")
ax5.set_xlabel("Corpus Source", labelpad=10)
ax5.set_ylabel("Mean Kohlberg Stage (0-6)", labelpad=10)
ax5.grid(axis="y", linestyle="--", alpha=0.5)
ax5.set_axisbelow(True)

for bar in ax5.patches:
    yval = bar.get_height()
    if yval > 0:
        ax5.text(bar.get_x() + bar.get_width()/2.0, yval - 0.3 if yval > 0.5 else yval + 0.1, 
                 f"{yval:.2f}", ha="center", va="center", color="white" if yval > 0.5 else "black", fontweight="bold")

sns.despine(ax=ax5)

# ----------------- Plot 6: Heatmap: Kohlberg Stage vs. Moral Foundations -----------------
ax6 = axes[2, 1]
heatmap_data = []
for cat in mfd_categories:
    if cat in scored_df.columns:
        cat_df = scored_df[scored_df[cat] == True]
        mean_c4 = cat_df[cat_df["source"] == "C4"]["kohlberg_stage"].mean() if len(cat_df[cat_df["source"] == "C4"]) > 0 else np.nan
        mean_reddit = cat_df[cat_df["source"] == "Reddit"]["kohlberg_stage"].mean() if len(cat_df[cat_df["source"] == "Reddit"]) > 0 else np.nan
        heatmap_data.append({
            "Moral Foundation": cat,
            "C4": mean_c4,
            "Reddit": mean_reddit
        })
if heatmap_data:
    heatmap_df = pd.DataFrame(heatmap_data).set_index("Moral Foundation")
    sns.heatmap(
        heatmap_df,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        cbar_kws={'label': 'Mean Kohlberg Stage (0-6)'},
        linewidths=.8,
        linecolor="#e0e0e0",
        annot_kws={"size": 11, "weight": "bold"},
        ax=ax6
    )
    ax6.set_title("Moral Foundations vs. Mean Kohlberg Stage", pad=15, fontweight="bold")
    ax6.set_xlabel("Corpus Source", labelpad=10)
    ax6.set_ylabel("Moral Foundation Category", labelpad=10)
else:
    ax6.text(0.5, 0.5, "No Moral Foundation columns found.", 
             ha="center", va="center", transform=ax6.transAxes)
    ax6.set_title("Moral Foundations vs. Kohlberg Stage (No Data)", pad=15, fontweight="bold")

sns.despine(ax=ax6, left=True, bottom=True)

plt.suptitle("Comparative Moral Reasoning Analysis: C4 vs. Reddit", y=0.99, fontsize=18, fontweight="bold")
plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.savefig("kohlberg_evaluation_results.png", dpi=300, bbox_inches="tight")
plt.savefig("kohlberg_evaluation_results.pdf", format="pdf", bbox_inches="tight")
plt.show()
